# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 clinical dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined in Croissant schema format and accessed via:
<br>
`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import matplotlib.pyplot as plt

# Dataset URL for FAIR^2
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("Dataset Name: ", metadata.name)
print("Dataset Description: ", metadata.description)

## 2. Data Overview
List available record sets, fields, and IDs to understand what tabular datasets are present.

**Note:** All references to entities are by their `@id` attributes.

In [ ]:
# Examine available record sets in FAIR^2 dataset
record_sets = dataset.record_sets
print("Record sets available:")
for rs in record_sets:
    print(f"- {rs['@id']} | {rs.get('name', '[no name]')}")

# For each record set, list its fields and columns (by @id)
for rs in record_sets:
    print(f"\nRecord set: {rs['@id']} ({rs.get('name', '[no name]')})")
    fields = rs.get('field', [])
    columns = rs.get('column', [])
    print("  Fields:")
    for f in fields:
        print(f"    - {f['@id']} | {f.get('name','[no name]')}")
    print("  Columns:")
    for c in columns:
        print(f"    - {c['@id']} | {c.get('name','[no name]')}")

## 3. Data Extraction
Extract data from each record set using its `@id` and load into Pandas DataFrames.

**Example:** Below, we load all record sets found in the overview step.

In [ ]:
# Gather the @id for each RecordSet from the previous step
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nColumns in {record_set_id}: {df.columns.tolist()}")
    print(df.head())

# Pick first record set for more detailed exploration
if len(record_set_ids) > 0:
    first_record_set_id = record_set_ids[0]
else:
    first_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply key data processing steps, such as filtering and normalization, using actual field and column `@id` from the dataset.

**Example Steps:**
- Filter for numeric field values above threshold
- Normalize a numeric column
- Group records by a key field

In [ ]:
# Example EDA on the first record set, if available
if first_record_set_id:
    df = dataframes[first_record_set_id]
    print(f"\nPerforming EDA for Record Set @id: {first_record_set_id}")

    # Identify which columns are numeric
    numeric_cols = df.select_dtypes(include=['float64','int64']).columns.tolist()
    print("Numeric columns:", numeric_cols)
    if len(numeric_cols) > 0:
        numeric_field = numeric_cols[0] # Use first numeric column for demo
        threshold = df[numeric_field].mean() # Use mean as threshold
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Grouping by a key field (select string-type column if available)
        string_cols = df.select_dtypes(include=["object"]).columns.tolist()
        if len(string_cols) > 0:
            group_field = string_cols[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped data by {group_field}:")
            print(grouped_df.head())
else:
    print("No record sets loaded for EDA.")

## 5. Visualization
Show example visualizations such as histograms or scatter plots for numeric fields in the dataset.

In [ ]:
# Visualize the distribution of the numeric field in the first record set
if first_record_set_id and len(numeric_cols) > 0:
    plt.figure(figsize=(8,4))
    df[numeric_field].hist(bins=10)
    plt.title(f'Distribution of {numeric_field} in {first_record_set_id}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # Scatter plot for numeric field vs another if available
    if len(numeric_cols) > 1:
        plt.figure(figsize=(6,6))
        plt.scatter(df[numeric_cols[0]], df[numeric_cols[1]])
        plt.title(f'Scatter Plot: {numeric_cols[0]} vs {numeric_cols[1]}')
        plt.xlabel(numeric_cols[0])
        plt.ylabel(numeric_cols[1])
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load the FAIR^2 dataset using mlcroissant, explored the available tables (record sets), extracted data using their `@id`s, performed basic filtering and normalization, and visualized key numeric fields. These steps make it possible to reproducibly analyze datasets following the Croissant schema.

Key takeaways:
- All dataset entities (record sets, fields, columns) are referenced by their `@id`.
- `mlcroissant` enables direct loading and manipulation of FAIR datasets.
- Proper handling ensures robust and reproducible biomedical/clinical data exploration.
